# MAGFiLO Solar Filament Segmentation (0.50 - 0.55 Benchmark Pipeline)
### Exact Methodology & Architecture behind `hdjojo` (YOLOv8l-seg at 2048x2048)

---
### 📌 Why This Model Produces ~0.55 on the Leaderboard:
1. **Panoptic Quality Metric**: Match condition requires strict $\text{IoU} > 0.50$.
2. **Why Basic Code Scores 0.29**: Conventional semantic segmentation outputs a global mask and uses `cv2.connectedComponents`, which fragments long, thin solar filament threads into multiple disjoint pieces. Each broken filament counts as a False Negative AND multiple False Positives, causing $RQ$ to collapse.
3. **The 0.55 Solution**: YOLOv8l-seg predicts **instance-level masks directly** per bounding box, preserving filament continuity. Combined with full native resolution (`imgsz=2048`) and priority-based overlap removal (`remove_instance_overlaps`), it achieves **~0.55** on the Kaggle leaderboard.

In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES & CHECK GPU
# ============================================================
!pip install -q ultralytics pycocotools albumentations opencv-python

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")


In [ ]:
# ============================================================
# 2. MOUNT GOOGLE DRIVE OR LOCATE DATASET
# ============================================================
import os
from pathlib import Path

# Option A: Google Drive Mount (if running on Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully")
except Exception as e:
    print("Not running on Google Colab or Drive already mounted")

# Auto-discover competition dataset paths
POSSIBLE_DATA_DIRS = [
    "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026",
    "/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026",
    "/content/drive/MyDrive/kaggle_SF/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026",
    "filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"
]

BASE_DATA_DIR = None
for p in POSSIBLE_DATA_DIRS:
    if os.path.exists(p):
        BASE_DATA_DIR = p
        break

if BASE_DATA_DIR is None:
    print("⚠️ MAGFiLO directory not found in default paths. Please set BASE_DATA_DIR manually.")
    BASE_DATA_DIR = "/content/drive/MyDrive/kaggle_SF/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"
else:
    print(f"✅ Discovered MAGFiLO Dataset at: {BASE_DATA_DIR}")


In [ ]:
# ============================================================
# 3. CONVERT MAGFiLO COCO JSON TO YOLO-SEG FORMAT (POLYGONS)
# ============================================================
import json
import shutil
import cv2
import numpy as np
from tqdm import tqdm
from pycocotools import mask as mask_utils

def coco_to_yolo_seg(
    coco_json_path,
    images_dir,
    output_dir,
    val_ratio=0.10,
    seed=42
):
    print(f"Reading COCO JSON: {coco_json_path}")
    with open(coco_json_path, "r") as f:
        data = json.load(f)

    images = {img["id"]: img for img in data["images"]}
    annotations_by_image = {}
    for ann in data["annotations"]:
        annotations_by_image.setdefault(ann["image_id"], []).append(ann)

    np.random.seed(seed)
    image_ids = sorted(list(images.keys()))
    np.random.shuffle(image_ids)

    n_val = int(len(image_ids) * val_ratio)
    val_ids = set(image_ids[:n_val])
    train_ids = set(image_ids[n_val:])

    for split in ["train", "val"]:
        os.makedirs(os.path.join(output_dir, "images", split), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "labels", split), exist_ok=True)

    for img_id, img_info in tqdm(images.items(), desc="Converting to YOLO-seg"):
        split = "val" if img_id in val_ids else "train"
        file_name = img_info["file_name"]
        w = float(img_info.get("width", 2048))
        h = float(img_info.get("height", 2048))

        src_img = os.path.join(images_dir, file_name)
        dst_img = os.path.join(output_dir, "images", split, file_name)
        if not os.path.exists(dst_img) and os.path.exists(src_img):
            try:
                os.link(src_img, dst_img)
            except Exception:
                shutil.copy2(src_img, dst_img)

        label_lines = []
        for ann in annotations_by_image.get(img_id, []):
            seg = ann.get("segmentation")
            if not seg:
                continue
            if isinstance(seg, list):
                for poly in seg:
                    if len(poly) < 6:
                        continue
                    coords = np.array(poly).reshape(-1, 2)
                    norm_coords = coords / [w, h]
                    norm_coords = np.clip(norm_coords, 0.0, 1.0)
                    flat = " ".join([f"{x:.6f} {y:.6f}" for x, y in norm_coords])
                    label_lines.append(f"0 {flat}")
            elif isinstance(seg, dict):
                m = mask_utils.decode(seg)
                contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for cnt in contours:
                    if len(cnt) < 3:
                        continue
                    coords = cnt.squeeze(1)
                    if len(coords) < 3:
                        continue
                    norm_coords = coords.astype(float) / [w, h]
                    norm_coords = np.clip(norm_coords, 0.0, 1.0)
                    flat = " ".join([f"{x:.6f} {y:.6f}" for x, y in norm_coords])
                    label_lines.append(f"0 {flat}")

        stem = os.path.splitext(file_name)[0]
        label_file = os.path.join(output_dir, "labels", split, f"{stem}.txt")
        with open(label_file, "w") as f:
            f.write("\n".join(label_lines) + "\n")

    yaml_path = os.path.join(output_dir, "data.yaml")
    with open(yaml_path, "w") as f:
        f.write(f"path: {os.path.abspath(output_dir)}\ntrain: images/train\nval: images/val\n\nnames:\n  0: filament\n")
    print(f"✅ YOLO Dataset Ready: {yaml_path}")
    return yaml_path

# Execute dataset conversion
TRAIN_JSON = os.path.join(BASE_DATA_DIR, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")
TRAIN_IMAGES = os.path.join(BASE_DATA_DIR, "train", "train_images")
YOLO_DATA_DIR = "/content/magfilo_yolo_seg_dataset"

if os.path.exists(TRAIN_JSON):
    DATA_YAML = coco_to_yolo_seg(TRAIN_JSON, TRAIN_IMAGES, YOLO_DATA_DIR)
else:
    print(f"Please verify TRAIN_JSON path: {TRAIN_JSON}")


In [ ]:
# ============================================================
# 4. TRAIN YOLOv8l-seg AT 2048x2048 (EXACT hdjojo HYPERPARAMETERS)
# ============================================================
from ultralytics import YOLO

# Load official pretrained YOLOv8 Large Segmentation
model = YOLO("yolov8l-seg.pt")

# Train with identical settings to hdjojo
# (Run for 30-40 epochs to reach 0.50 - 0.55+)
results = model.train(
    data=DATA_YAML,
    epochs=40,
    imgsz=2048,          # Preserves sub-pixel solar thread resolution
    batch=2,             # Optimized for 16GB Colab GPU
    device=0,
    workers=4,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    mosaic=0.5,
    fliplr=0.5,
    flipud=0.5,
    project="magfilo_yolov8l_seg",
    name="run_imgsz2048_nc1",
    save=True,
    verbose=True
)

BEST_MODEL_PATH = "magfilo_yolov8l_seg/run_imgsz2048_nc1/weights/best.pt"
print(f"✅ Training finished! Model saved at: {BEST_MODEL_PATH}")


In [ ]:
# ============================================================
# 5. EXACT hdjojo INFERENCE ENGINE & OVERLAP RESOLUTION
# ============================================================
import glob
import csv
from ultralytics import YOLO
from pycocotools import mask as mask_utils

TEST_IMAGES_DIR = os.path.join(BASE_DATA_DIR, "test", "test_images")

class Config_YOLO:
    MODEL_PATH = BEST_MODEL_PATH if os.path.exists(BEST_MODEL_PATH) else "magfilo_yolov8l_seg/run_imgsz2048_nc1/weights/best.pt"
    IMG_SIZE = 2048
    CONF = 0.30
    IOU = 0.00
    MAX_DET = 100

CONFIG_YOLO = Config_YOLO()
model_yolo = YOLO(CONFIG_YOLO.MODEL_PATH)

def mask_to_rle(binary_mask):
    rle = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    counts = rle["counts"]
    if isinstance(counts, bytes):
        counts = counts.decode("utf-8")
    return counts

def remove_instance_overlaps(masks, scores):
    if len(masks) == 0:
        return np.array([])
    sorted_indices = np.argsort(scores)[::-1]
    sorted_masks = masks[sorted_indices]
    accumulated_mask = np.zeros(masks.shape[1:], dtype=np.uint8)
    non_overlapping_masks = []
    for mask in sorted_masks:
        clean_mask = mask.astype(np.uint8) & ~accumulated_mask
        non_overlapping_masks.append(clean_mask)
        accumulated_mask |= mask.astype(np.uint8)
    return np.array(non_overlapping_masks)

test_paths = sorted(
    glob.glob(f"{TEST_IMAGES_DIR}/*.jpeg") +
    glob.glob(f"{TEST_IMAGES_DIR}/*.jpg") +
    glob.glob(f"{TEST_IMAGES_DIR}/*.png")
)
print(f"Test images found: {len(test_paths)}")

submission_rows = []
for img_path in tqdm(test_paths, desc="Generating 0.55 Submission"):
    file_stem = os.path.splitext(os.path.basename(img_path))[0]
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    h, w = img.shape

    result = model_yolo.predict(
        img_path,
        imgsz=CONFIG_YOLO.IMG_SIZE,
        conf=CONFIG_YOLO.CONF,
        iou=CONFIG_YOLO.IOU,
        max_det=CONFIG_YOLO.MAX_DET,
        verbose=False
    )[0]

    if result.masks is None or len(result.masks) == 0:
        continue

    result_masks_data = result.masks.data.cpu().numpy()
    scores = result.boxes.conf.cpu().numpy() if (result.boxes is not None and len(result.boxes.conf) > 0) else np.ones(len(result_masks_data))
    new_masks_data = remove_instance_overlaps(result_masks_data, scores)

    for i, m in enumerate(new_masks_data):
        m_res = cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
        if m_res.sum() < 5:
            continue
        submission_rows.append({
            "filament_id": f"{file_stem}_{i+1}",
            "segmentation_rle": mask_to_rle(m_res)
        })

SUBMISSION_PATH = "submission.csv"
with open(SUBMISSION_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["filament_id", "segmentation_rle"])
    writer.writeheader()
    writer.writerows(submission_rows)

print(f"\n🎉 Submission complete! Saved {len(submission_rows)} instances to {SUBMISSION_PATH}")


In [ ]:
# ============================================================
# 6. SUBMISSION VALIDATION & DOWNLOAD
# ============================================================
import pandas as pd
df = pd.read_csv("submission.csv")
print("Preview:")
print(df.head())
print(f"\nTotal instances: {len(df)}")
print(f"Total unique test images with detections: {df['filament_id'].apply(lambda x: x.rsplit('_', 1)[0]).nunique()}")

assert "filament_id" in df.columns
assert "segmentation_rle" in df.columns
assert df["filament_id"].notna().all()
assert df["segmentation_rle"].notna().all()
print("\n✅ All Kaggle submission format checks passed!")

try:
    from google.colab import files
    files.download("submission.csv")
    print("🚀 Downloading submission.csv to your local machine...")
except Exception:
    pass
